In [1]:
import json

import requests
from IPython.display import SVG, display
import plotly.graph_objects as go
import plotly.io as pio

import pandas as pd
import geopandas as gpd

In [2]:
pd.options.display.float_format = '{:20,.2f}'.format
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_columns', 1000)
pd.set_option('display.width', 1000)

In [3]:
country_gpd = gpd.read_file(r"UN_Countries_Simplified.shp")

In [4]:
map_data = gpd.read_file("../../data/ouput/v2/spatial/countries_edits_ai_20_25.geojson")

In [5]:
result_all_big = pd.read_csv("../../data/ouput/v2/spatial/allyear_indicator_csv.csv")

In [6]:
countries = ["ALB", "BGD", "BOL", "BTN", "BRN",  "CYP", "ECU", "ETH", "B35", "GRC", "HUN", "IND", "KEN",  "KOR",
              "MAR", "MKD", "MNE", "NGA", "NZ1", "PNG", "SAU",  "TUR", "TZA", "US1", "VNM"
            ]

In [8]:
result_all_big["contributors_prolific_pct_ai"]=result_all_big["contributors_prolific_share_ai"]*100
result_all_big["contributors_casual_pct_ai"]=result_all_big["contributors_casual_share_ai"]*100
result_all_big["contributors_inactive_pct_ai"]=result_all_big["contributors_inactive_share_ai"]*100

In [9]:
result_all_big["highway_changeset_pct"]=result_all_big["highway_changesets_share"]*100

In [10]:
result_all_big["building_changeset_pct"]=result_all_big["building_changesets_share"]*100

In [11]:
result_all_big["changesets_corporate_pct"]=result_all_big["changesets_corporate_share"]*100
result_all_big["changesets_humanitarian_pct"]=result_all_big["changesets_humanitarian_share"]*100

In [12]:
share_df = result_all_big[result_all_big['country'].isin(["ALB", "BGD", "BOL", "BTN", "BRN",  "CYP", "ECU", "ETH", "B35", "GRC", "HUN", "IND", "KEN",  "KOR",
              "MAR", "MKD", "MNE", "NGA", "NZ1", "PNG", "SAU",  "TUR", "TZA", "US1", "VNM"])]

In [13]:
share_df = share_df[['country', 'total_new_contrib', 'changesets_corporate_pct',  'changesets_humanitarian_pct',
               'building_changeset_pct', 'highway_changeset_pct','pct_trend_active', 'pct_trend_changesets', 'pct_trend_new_contrib',
                    'contributors_prolific_pct_ai', 'contributors_casual_pct_ai','contributors_inactive_pct_ai' ]]

#### Total numbers
Clean from basic dataset

In [14]:
def absolutes_data(country):
    #the updated indicator list from Creating_final_fixed_idnicator_csv.ipynb
    df = pd.read_csv(rf"..\..\data\ouput\country_analysis\v5_indicators\{country}\{country}_monthly_stats.csv")
    total = df.loc[(df["months"] <="2025-12" ) & (df["months"] >"2019-12" )]
    total = total[["Edits", "Contributors", "Changesets", "EditsAI", "ChangesetsAI", "ContributorsAI"]].sum()
    total_df = total.to_frame().T
    total_df["editsAI_share"] = (total_df["EditsAI"]/total_df["Edits"])*100
    total_df["changesetsAI_share"] = (total_df["ChangesetsAI"]/total_df["Changesets"])*100
    total_df["contributorsAI_share"] = (total_df["ContributorsAI"]/total_df["Contributors"])*100
    total_df["country"] =f"{country}"
    return total_df

In [15]:
dfs_total = []

for c in countries:
    dfb = absolutes_data(c)
    dfs_total.append(dfb)

result_all_totals = pd.concat(dfs_total, ignore_index=True)

In [38]:
result_all_totals.sort_values(by="ChangesetsAI").tail(5)

,Edits,Contributors,Changesets,EditsAI,ChangesetsAI,ContributorsAI,editsAI_share,changesetsAI_share,contributorsAI_share,country
21,"12,883,763.00","42,924.00","686,481.00","2,421,896.00","100,631.00","7,954.00",18.80,14.66,18.53,TUR
22,"15,790,608.00","26,565.00","604,545.00","6,343,443.00","171,731.00","4,817.00",40.17,28.41,18.13,TZA
24,"10,506,088.00","22,072.00","981,365.00","3,232,039.00","210,867.00","1,103.00",30.76,21.49,5.00,VNM
11,"37,108,878.00","112,702.00","3,150,822.00","13,107,416.00","893,481.00","5,066.00",35.32,28.36,4.50,IND
23,"255,331,068.00","379,262.00","13,442,077.00","44,317,899.00","1,748,120.00","37,945.00",17.36,13.00,10.00,US1


In [25]:
country_df = result_all_totals.merge(share_df, on="country")

In [40]:
country_df[["EditsAI","ChangesetsAI","ContributorsAI"]].sum()

EditsAI                 80,157,843.00
ChangesetsAI             3,361,459.00
ContributorsAI              68,825.00
dtype: float64

In [137]:
country_df = country_df.merge(map_data[["ADMIN", "ADM0_A3"]], left_on ="country", right_on="ADM0_A3", how="left")

In [138]:
country_df.to_csv(r"\data\ouput\appednix_overview.csv", sep=';', decimal=',', index=False)

#### Comparison 25 countries to toal AI edits

In [18]:
df_ai_total = pd.read_csv("../../data/ouput/v2/monthly_ai_stats.csv").drop(columns={"Unnamed: 0"})

In [20]:
df_ai_total = df_ai_total[(df_ai_total["months"] >= "2020-01") & (df_ai_total["months"] <= "2025-12")]

In [22]:
df_ai_total[["Contributors", "Edits", "Changesets"]].sum()

Contributors        54469
Edits           560049571
Changesets        3741766
dtype: int64

In [28]:
country_df[["EditsAI","ChangesetsAI","ContributorsAI"]].sum()

EditsAI                 80,157,843.00
ChangesetsAI             3,361,459.00
ContributorsAI              68,825.00
dtype: float64

In [41]:
country_df[["EditsAI","ChangesetsAI","ContributorsAI"]].sort_values(by="ChangesetsAI").tail(5).sum()

EditsAI                 69,422,693.00
ChangesetsAI             3,124,830.00
ContributorsAI              56,885.00
dtype: float64

In [30]:
(3361459/3741766)*100

89.83616292413797

In [35]:
(1748120/3741766)*100

46.71911605375643

In [42]:
(3124830/3741766)*100

83.51217045641015